# DINO mu-probe: interactive 3D plots

Companion to `run_probe.py` (which writes static PNGs). This notebook reuses its
functions directly so the two never drift apart, and renders the same views as
rotatable Plotly 3D scatters: PCA, t-SNE, and the fitted classifier direction.

Edit `MU_LOW` / `MU_HIGH` below to change where the low/high-mu boundary sits
(see `derive_labels` docstring for the caveat: `features.npz` only has points
with mu<=0.6 or mu>=0.85 cached, so thresholds outside that range only relabel
or drop cached points, they can't recover the unextracted middle band).

In [1]:
import importlib.util
import pathlib

import numpy as np
import plotly.graph_objects as go
import yaml
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE


def find_repo_root(start="."):
    p = pathlib.Path(start).resolve()
    while not (p / "AGENTS.md").exists():
        p = p.parent
    return p


REPO_ROOT = find_repo_root()
PROBE_DIR = REPO_ROOT / "side_experiments/dino_mu_probe"

spec = importlib.util.spec_from_file_location("run_probe", PROBE_DIR / "run_probe.py")
rp = importlib.util.module_from_spec(spec)
spec.loader.exec_module(rp)

In [2]:
MU_LOW, MU_HIGH = 0.6, 0.85
CFG = REPO_ROOT / "training/traversability_network/configs/grandtour_lite.yaml"

d = np.load(PROBE_DIR / "output/features.npz")
X, mission, mu = d["features"], d["mission"], d["mu"]

keep, y = rp.derive_labels(mu, MU_LOW, MU_HIGH)
print(f"dropping {(~keep).sum()} points outside [mu<={MU_LOW} or mu>={MU_HIGH}]" if not keep.all() else "no points dropped")
X, mission, mu, y = X[keep], mission[keep], mu[keep], y[keep]
print(f"{len(y)} points, {int(y.sum())} low-mu, model {d['model']}")

cfg = yaml.safe_load(open(CFG))
fit = rp.fit_official_split(X, y, mu, mission, cfg["DATASET"]["TRAIN_MISSIONS"], cfg["DATASET"]["VALID_MISSIONS"])
auc = rp.eval_official_split(fit)

no points dropped
7024 points, 1228 low-mu, model dinov2_vitb14_reg



official train->valid split (train=[np.str_('ALB-1'), np.str_('ALB-2'), np.str_('CYN-1'), np.str_('HIL-1'), np.str_('HOEB-1'), np.str_('KAEB-2'), np.str_('KAEB-3'), np.str_('LMB-1'), np.str_('RIV-1'), np.str_('TRIM-1')], valid=[np.str_('ALB-3'), np.str_('CYN-2'), np.str_('HOEB-2'), np.str_('LMB-2')])
AUROC 0.795  AP 0.372  AP-base 0.162  n_low 199  n 1227


## 3-component PCA

Unsupervised: the three axes of largest overall variance, which need not align
with the mu-discriminative direction. Rotate freely to look for structure the
static multi-angle PNG might have missed between its fixed viewpoints.

In [3]:
p3 = PCA(3).fit_transform(X)

fig = go.Figure(go.Scatter3d(
    x=p3[:, 0], y=p3[:, 1], z=p3[:, 2],
    mode="markers",
    marker=dict(size=3, color=mu, colorscale="Viridis", opacity=0.8, colorbar=dict(title="mu")),
    text=[f"mission={m}<br>mu={v:.2f}" for m, v in zip(mission, mu)],
    hoverinfo="text",
))
fig.update_layout(
    title="frozen DINO patch features (3-component PCA)",
    scene=dict(xaxis_title="PC1", yaxis_title="PC2", zaxis_title="PC3"),
    height=700,
)
fig.show()

## 3-component t-SNE

Nonlinear, preserves local neighborhoods instead of global variance -- can
reveal low-mu clustering that PCA spreads across low-variance directions.

In [4]:
Xs = (X - X.mean(0)) / (X.std(0) + 1e-8)
emb3 = TSNE(n_components=3, init="pca", perplexity=30, random_state=0).fit_transform(Xs)

fig = go.Figure(go.Scatter3d(
    x=emb3[:, 0], y=emb3[:, 1], z=emb3[:, 2],
    mode="markers",
    marker=dict(size=3, color=mu, colorscale="Viridis", opacity=0.8, colorbar=dict(title="mu")),
    text=[f"mission={m}<br>mu={v:.2f}" for m, v in zip(mission, mu)],
    hoverinfo="text",
))
fig.update_layout(
    title="frozen DINO patch features (3-component t-SNE)",
    scene=dict(xaxis_title="t-SNE 1", yaxis_title="t-SNE 2", zaxis_title="t-SNE 3"),
    height=700,
)
fig.show()

## Classifier direction + 2 PCA components

x = the fitted logistic probe's own decision axis (the exact direction AUROC is
computed along); y/z = the top 2 unsupervised PCA components, for spread. Faint
points are the train fold the classifier was fit to (expect a clean split at
x=0); solid points are the held-out valid fold -- how much of that split
survives there is the honest AUROC story (see the plot title for the current value).

In [5]:
pc2 = PCA(n_components=2).fit(fit["Xtr_n"])
x_tr = fit["clf"].decision_function(fit["Xtr_n"])
x_te = fit["clf"].decision_function(fit["Xte_n"])
yz_tr = pc2.transform(fit["Xtr_n"])
yz_te = pc2.transform(fit["Xte_n"])

fig = go.Figure()
fig.add_trace(go.Scatter3d(
    x=x_tr, y=yz_tr[:, 0], z=yz_tr[:, 1], mode="markers", name="train (fit on this)",
    marker=dict(size=2, color=fit["mu_tr"], colorscale="Viridis", opacity=0.3, showscale=False),
    text=[f"mu={v:.2f}" for v in fit["mu_tr"]], hoverinfo="text",
))
fig.add_trace(go.Scatter3d(
    x=x_te, y=yz_te[:, 0], z=yz_te[:, 1], mode="markers", name="valid (held out)",
    marker=dict(size=4, color=fit["mu_te"], colorscale="Viridis", opacity=0.9, colorbar=dict(title="mu")),
    text=[f"mu={v:.2f}" for v in fit["mu_te"]], hoverinfo="text",
))
fig.update_layout(
    title=f"DINO features projected onto the fitted low-mu classifier direction (AUROC={auc:.3f})",
    scene=dict(
        xaxis_title="classifier direction (logit, >0 -> predicts low-mu)",
        yaxis_title="PC1", zaxis_title="PC2",
    ),
    height=750,
)
fig.show()